# M2 orientation classifiers + net-extraction check (Kaggle)

Runs two things at the same time:

| Where | Job |
|---|---|
| GPU (or CPU if none) | 8 orientation / mirroring classifiers, one after another |
| CPU | net extraction on the 37 Image2Net detection images that copy evaluation circuits, with their hand-drawn boxes, scored against the golden netlists; once cutting plain wire crossings, once not |
| after training | the same check again, with orientations from the new classifiers |

**Before running**

1. On your machine: `uv run python scripts/prepare_m2_data.py --small-zip-only`
   writes `data/m2_kaggle_small.zip` (about 40 MB).
2. Kaggle → *Datasets* → *New dataset* → upload it, keep it private.
3. In this notebook: *Add input* → that dataset; *Settings* → Accelerator
   **GPU T4 x2** or **GPU P100** (or *None*: CPU only, slower), Internet **on**.
4. **Save Version → Save & Run All**. Results: `results_m2_small.zip` under *Output*.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# The small dataset is the one with netting_eval/ next to code/.
roots = [
    p.parent.parent
    for p in Path("/kaggle/input").rglob("CODE_COMMIT")
    if (p.parent.parent / "netting_eval").exists()
]
assert roots, "Attach the m2_kaggle_small dataset (Add input) first."
SRC = roots[0]
COMMIT = (SRC / "code" / "CODE_COMMIT").read_text().strip()
print("dataset root:", SRC)
print("code commit :", COMMIT)

WORK = Path("/kaggle/working")
CODE, DATA, EVAL = WORK / "code", WORK / "datasets", WORK / "netting_eval"
RESULTS = WORK / "results" / "m2"
for target, source in ((CODE, SRC / "code"), (DATA, SRC / "datasets"),
                       (EVAL, SRC / "netting_eval")):
    if not target.exists():
        shutil.copytree(source, target)
RESULTS.mkdir(parents=True, exist_ok=True)
if (CODE / "UNCOMMITTED.diff").exists():  # record what the -dirty commit means
    shutil.copy(CODE / "UNCOMMITTED.diff", RESULTS / "UNCOMMITTED.diff")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics==8.4.192", "scikit-learn==1.9.1"], check=True)
import torch  # noqa: E402

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("orientation device:", torch.cuda.get_device_name(0) if DEVICE == "0" else "cpu")

In [ ]:
ENV = dict(os.environ, PYTHONPATH=str(CODE / "src"), VCA_CODE_COMMIT=COMMIT)


def q(args):
    return " ".join(f"'{a}'" for a in args)


# Orientation: horizontal flips off (a flip turns "l" into "r" and swaps the
# mirror state, so the default 0.5 would mislabel half the flipped samples).
tasks = sorted(p.name for p in (DATA / "orientation").iterdir() if p.is_dir())
orientation = [
    [sys.executable, "-m", "vca.training", "classify",
     "--data", str(DATA / "orientation" / t), "--name", f"orientation_{t}",
     "--device", DEVICE, "--epochs", "60", "--batch", "64", "--workers", "2",
     "--fliplr", "0", "--output-dir", str(RESULTS)]
    for t in tasks
]
# Net extraction check (CPU, one GED alignment per circuit). Runs now with
# conventional orientations, and again after training with the classifiers.


def netting_run(variant, flags):
    return [sys.executable, str(CODE / "scripts" / "eval_netting_oracle.py"),
            "--manifest", str(EVAL / "manifest.json"),
            "--detection-dir", str(EVAL / "device_identification"),
            "--golden-dir", str(EVAL / "golden"),
            "--output-dir", str(RESULTS / f"netting_oracle_{variant}"),
            "--workers", "3", *flags]


CUT, NOCUT = [], ["--no-plain-crossings"]
PREDICTED = ["--orientation-results", str(RESULTS)]
netting = [netting_run("cut", CUT), netting_run("nocut", NOCUT)]
orientation += [netting_run("cut_oriented", CUT + PREDICTED),
                netting_run("nocut_oriented", NOCUT + PREDICTED)]

procs = {}
for name, commands in (("orientation", orientation), ("netting", netting)):
    log = RESULTS / f"{name}.log"
    chain = " ; ".join(q(c) for c in commands)  # keep going if one task fails
    procs[name] = subprocess.Popen(["bash", "-c", chain], env=ENV,
                                   stdout=open(log, "w"), stderr=subprocess.STDOUT)
    print(f"{name}: {len(commands)} job(s), log {log}")

In [ ]:
# Wait for both, printing the latest progress line every 5 minutes.
def last_line(path):
    text = path.read_text(errors="replace").replace("\r", "\n").strip().splitlines()
    return text[-1][-160:] if text else ""


while any(p.poll() is None for p in procs.values()):
    for name, p in procs.items():
        code = p.poll()
        state = "running" if code is None else f"exit {code}"
        latest = last_line(RESULTS / f"{name}.log")
        print(f"[{time.strftime('%H:%M')}] {name} {state}: {latest}")
    time.sleep(300)

for name, p in procs.items():
    print(f"{name} finished with exit code {p.returncode}")

In [ ]:
# Held-out results, then package everything.
for metrics in sorted(RESULTS.glob("orientation_*/*/metrics.json")):
    m = json.loads(metrics.read_text())
    recall = {k: v["recall"] for k, v in m["per_class"].items()}
    print(f"{metrics.parent.parent.name:28s} top1={m['top1']}  recall={recall}")
for metrics in sorted(RESULTS.glob("netting_oracle_*/*/metrics.json")):
    m = json.loads(metrics.read_text())
    print(metrics.parent.parent.name, json.dumps(m["all"]))

archive = shutil.make_archive(str(WORK / "results_m2_small"), "zip", WORK / "results")
print(archive, round(Path(archive).stat().st_size / 1e6), "MB")